# Customer Segmentation using K-Means

**Objective:**

The objective is to segment customers based on spending behavior and income using K-Means clustering in order to design targeted marketing strategies for each group.

## STEP 1 — Install & Import

If you don't have the required libraries installed, run the cell below.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn'])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set(style="whitegrid")
%matplotlib inline

## STEP 2 — Load Dataset

Make sure `Mall_Customers.csv` is in this folder (`task-2-customer-segmentation`).

In [ ]:
df = pd.read_csv('Mall_Customers.csv')
df.head()

## STEP 3 — Basic EDA

Quick checks: info, describe, missing values, and a couple of plots.

In [ ]:
df.info()

df.describe()

df.isnull().sum()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='Gender', data=df)
plt.title('Gender Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(6,5))
sns.scatterplot(x='Annual Income (k$)', y='Spending Score (1-100)', data=df)
plt.title('Income vs Spending Score')
plt.show()

## STEP 4 — Feature Selection

We'll cluster using `Annual Income (k$)` and `Spending Score (1-100)`.

In [ ]:
X = df[['Annual Income (k$)', 'Spending Score (1-100)']].copy()
X.head()

## STEP 5 — Scale the Data

K-Means is distance-based, so scaling is important.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled[:5]

## STEP 6 — Elbow Method (Find Optimal K)

Compute WCSS (within-cluster sum of squares) for K=1..10 and plot the elbow.

In [ ]:
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(7,4))
plt.plot(range(1, 11), wcss, marker='o')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.title('Elbow Method')
plt.xticks(range(1,11))
plt.show()

## STEP 7 — Apply K-Means

Based on the elbow method we will use 5 clusters (common for this dataset).

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)
df.head()

## STEP 8 — Visualize Clusters

Scatter plot of `Annual Income` vs `Spending Score` colored by cluster.

In [ ]:
plt.figure(figsize=(7,6))
sns.scatterplot(x='Annual Income (k$)', y='Spending Score (1-100)', hue='Cluster', palette='Set1', data=df, s=60)
plt.title('Customer Segments')
plt.legend(title='Cluster')
plt.show()

## STEP 9 — PCA (Dimensionality Reduction)

Although we used two features, we include PCA for completeness.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]
plt.figure(figsize=(7,6))
sns.scatterplot(x='PCA1', y='PCA2', hue='Cluster', data=df, palette='Set1', s=60)
plt.title('PCA Visualization of Clusters')
plt.legend(title='Cluster')
plt.show()

## STEP 10 — Analyze Cluster Characteristics

Compute mean income and spending per cluster to understand each segment.

In [ ]:
cluster_summary = df.groupby('Cluster')[['Annual Income (k$)', 'Spending Score (1-100)']].mean().round(2)
cluster_counts = df['Cluster'].value_counts().sort_index()
cluster_summary['Count'] = cluster_counts.values
cluster_summary

## STEP 11 — Marketing Strategy (Interpretation)

Interpret the clusters from the `cluster_summary` above and design targeted strategies. Below are example recommendations — adapt them to the actual summary values shown by your run.

- **Cluster 0 — High Income, High Spending:** Premium customers. Offer loyalty rewards, premium bundles, and early access to new products.
- **Cluster 1 — High Income, Low Spending:** Wealthy but conservative. Use personalized premium offers and targeted messaging highlighting value and exclusivity.
- **Cluster 2 — Low Income, High Spending:** Impulsive or trend-driven buyers. Run discount-based campaigns and time-limited offers.
- **Cluster 3 — Low Income, Low Spending:** Budget-conscious segment. Promote basic value products and bundle discounts.
- **Cluster 4 — Mid Income, Mid Spending:** Stable/occasional buyers. Use seasonal promotions and cross-sell opportunities.

## Final Conclusion

K-Means clustering identified 5 distinct customer segments based on income and spending patterns. These segments enable targeted marketing strategies that can improve customer engagement and revenue optimization. Always validate strategies against the `cluster_summary` table and consider adding more features (age, gender, product preferences) for richer segmentation.